In [42]:
!pip install lenskit

In [43]:
from lenskit.algorithms import Recommender, item_knn as knn
from lenskit import batch, topn, util
from sklearn.model_selection import train_test_split
import pandas as pd
import joblib
import gzip
import json

In [44]:
# Function to down-sample the dataset
def downsample_data(ratings, percentage):
    sampled_ratings = ratings.sample(frac=percentage, random_state=1)
    return sampled_ratings

In [45]:
# Function to load the JSON data
def load_json_data(file_path):
    with gzip.open(file_path, 'r') as f:
        data = [json.loads(line) for line in f]
    return pd.DataFrame(data)

In [46]:
file_path = 'E:/Mechatronics_Uni Siegen Courses/Master Thesis/datasets/Amazon/Toys_and_Games_5.json.gz'
ratings = load_json_data(file_path)

In [47]:
ratings = ratings.rename(columns={'reviewerID': 'user', 'asin': 'item', 'overall': 'rating'})

In [48]:
ratings = ratings.dropna(subset=['rating'])

In [49]:
# Keep only the necessary columns
ratings = ratings[['user', 'item', 'rating']]

In [50]:
# Handle duplicate ratings by averaging them
ratings = ratings.groupby(['user', 'item']).agg({'rating': 'mean'}).reset_index()

In [51]:
# Down-sample the dataset to different percentages (adjust percentage as needed)
ratings = downsample_data(ratings, percentage=1.0)

In [52]:
# Split data into training, validation, and test sets (60%, 20%, 20%)
train_data, remaining_data = train_test_split(ratings, test_size=0.4, random_state=1)
valid_data, test_data = train_test_split(remaining_data, test_size=0.5, random_state=1)
print(train_data)
print(valid_data)
print(test_data)

                   user        item  rating
826557   A2S9SLRYLPGYZB  B00EQ0UQCM     5.0
978741   A33VI944SO2YCN  B00CG0CKVE     4.0
1682182   AU1RMMTXL0KV5  B00BY2ERIE     5.0
1228791  A3N8U4K6L9TMKX  B00EYTM7XG     5.0
751157   A2MGVQFHGIBOYH  B00GQBRNR0     5.0
...                 ...         ...     ...
1366527   A5H86XYSS0OU6  B00TXI1MZW     5.0
35107    A12RIBH3FE09MH  B00WE0OODI     5.0
560127   A27S4N0PJMZZ2Q  B00SUSUVSK     1.0
1039119  A38JVP8Y9TQ8GS  B00D8DK82G     4.0
39728    A136RY8WVROAMB  B00F2K5Z44     5.0

[1054492 rows x 3 columns]
                   user        item  rating
1238194   A3NYVOOT9FIQH  B001JTAZPG     5.0
1134870  A3FY7E4954X7E1  B00CS16KSU     5.0
844263   A2TPADE2JQV2VV  B00WHZ81WK     5.0
847870   A2TYULN0SL80OT  B007JWWL6I     5.0
423690   A1X54TQ4VTUCY8  B008Z1LBU4     5.0
...                 ...         ...     ...
1627209   APPQJACAV1EPO  B00JEYV048     5.0
263500   A1KT45EULSBC1F  B00B2B0I62     5.0
1111870   A3E7NTBUYLKVD  B007EA5WGQ     5.0
1659

In [53]:
# Initialize ItemItem recommender algorithm
algo_ii = knn.ItemItem(10)

In [54]:
# Training the model
fittable = util.clone(algo_ii)
fittable = Recommender.adapt(fittable)
fittable.fit(train_data)

In [55]:
# Save the model with a unique name
joblib.dump(fittable, 'itemitem_model.pkl')

['itemitem_model.pkl']

In [56]:
# Function to evaluate the algorithm 0n validation set
def evaluate_algorithm(algo, valid, Rec_Num):


    users = valid.user.unique()
    recs = batch.recommend(algo, users, Rec_Num)
    recs['Algorithm'] = 'ItemItem'

    # Compute nDCG on validation set
    rla = topn.RecListAnalysis()
    rla.add_metric(topn.ndcg)
    results = rla.compute(recs, valid)
    nDCG_mean = results.groupby('Algorithm').ndcg.mean().iloc[0]

    return nDCG_mean

In [57]:
Rec_Num = 10
#nDCG_validation = evaluate_algorithm(fittable, valid_data, Rec_Num)
#print("Validation nDCG @", Rec_Num, "for ItemItem:" ,nDCG_validation)

In [58]:
# Function to load the model and evaluate on test set
def evaluate_on_test_set(test,Rec_Num):
    # Load the model
    loaded_model = joblib.load('itemitem_model.pkl')

    # Evaluate on test set
    test_users = test.user.unique()
    test_recs = batch.recommend(loaded_model, test_users, Rec_Num)
    test_recs['Algorithm'] = 'ItemItem'

    # Compute nDCG on test set
    rla = topn.RecListAnalysis()
    rla.add_metric(topn.ndcg)
    test_results = rla.compute(test_recs, test)
    test_nDCG_mean = test_results.groupby('Algorithm').ndcg.mean().iloc[0]

    return test_nDCG_mean

In [59]:
# Evaluate on test set whenever needed
nDCG_test = evaluate_on_test_set(test_data,Rec_Num)
print(f"Test nDCG for ItemItem: {nDCG_test}")

Test nDCG for ItemItem: 0.002569847338297965


In [60]:
import os
print(os.getcwd())

C:\Users\ardal\Documents\Python Scripts\Jupyter Notebook
